In [1]:
from caftb_net import CAFTB_Net
from pathlib import Path

d:\Coding\Project\forensics_test\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = CAFTB_Net()

In [3]:
checkpoint_path = "D:/Coding/Project/forensics_test/weights/caftb-9.pth"

In [4]:
import torch

In [5]:
checkpoint = torch.load(checkpoint_path, map_location='cuda' if torch.cuda.is_available() else 'cpu', weights_only=False)
state_dict = checkpoint["model"] if "model" in checkpoint else checkpoint

In [6]:
model.load_state_dict(state_dict, strict=True)
model.eval()

CAFTB_Net(
  (cnn): FeatureListNet(
    (stem_conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (stem_norm1): GroupNormAct(
      32, 32, eps=1e-05, affine=True
      (drop): Identity()
      (act): ReLU(inplace=True)
    )
    (stem_conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (stem_norm2): GroupNormAct(
      32, 32, eps=1e-05, affine=True
      (drop): Identity()
      (act): ReLU(inplace=True)
    )
    (stem_conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (stem_pool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (stages_0): ResNetStage(
      (blocks): Sequential(
        (0): PreActBottleneck(
          (downsample): DownsampleAvg(
            (pool): Identity()
            (conv): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (norm): Identity()
          )
          (norm1): GroupNormAct(
    

In [7]:

import cv2
import numpy as np
import torch
from PIL import Image
from torchvision import transforms

# from ForensicHub.registry import MODELS, build_from_registry


def load_image(image_path: str, image_size: int = 512) -> torch.Tensor:
    transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

    image = Image.open(image_path).convert("RGB")
    return transform(image).unsqueeze(0)

In [8]:
image_size = 512
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
image_path = "D:/Coding/Project/forensics_test/images/tampered2.png"
output_path = "D:/Coding/Project/forensics_test/output/tampered2_heatmap.png"

In [9]:
image_tensor = load_image(image_path, image_size=image_size).to(device)
model = model.to(device)

In [10]:
# Pseudo-mask required by the model forward.
dummy_mask = torch.zeros(1, 1, image_size, image_size).to(device)

with torch.no_grad():
    output = model(image=image_tensor, mask=dummy_mask)
    pred_mask = output["pred_mask"][0, 0].detach().cpu().numpy()

In [11]:
def save_heatmap(image_path: str, pred_mask: np.ndarray, output_path: str) -> None:
    image = cv2.imread(image_path)
    image = cv2.resize(image, (pred_mask.shape[1], pred_mask.shape[0]))

    heatmap = (pred_mask * 255).astype(np.uint8)
    heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    overlay = cv2.addWeighted(image, 0.6, heatmap_color, 0.4, 0)
    cv2.imwrite(output_path, overlay)

In [12]:
Path(output_path).parent.mkdir(parents=True, exist_ok=True)
save_heatmap(image_path, pred_mask, output_path)

print(f"Saved heatmap to: {output_path}")
print(f"Suspicion score mean: {pred_mask.mean():.4f}")
print(f"Suspicion score max: {pred_mask.max():.4f}")

Saved heatmap to: D:/Coding/Project/forensics_test/output/tampered2_heatmap.png
Suspicion score mean: 0.0000
Suspicion score max: 0.0163
